# Session 3: EEG, from raw recording to RSA

A quick recap of where we have been so far:

  - [sess-1a](sess-1a.ipynb) covered neuroimaging file formats, NIfTI, BIDS, and atlases on the MNI152 template and the Harvard-Oxford atlas.
  - [sess-1b](sess-1b.ipynb) ran QC (MRIQC) and preprocessing (fMRIprep) on the **ds000117** face-recognition fMRI dataset.
  - [sess-2](sess-2.ipynb) ran first/second-level GLMs and functional connectivity on the **language-localizer demo** dataset, and closed with **RSA** on the **Kriegeskorte 92-image** fMRI data (Algonauts 2019 EVC and IT RDMs).

Today we switch modality. The lesson is organised around **three main themes**:

1. **Preprocessing** an EEG recording, turning a continuous, raw 64-channel BioSemi recording into clean, stimulus-locked trials (filtering, bad-channel repair, re-referencing, downsampling, epoching, artefact rejection).
2. **Event-related potentials (ERP)**: average those trials and inspect the **P300** in time and across the scalp.
3. **Time-resolved RSA**: re-meet sess-2 here. We pick up the **Kriegeskorte 92-image** dataset again, this time with the **MEG** RDMs from Cichy, Pantazis & Oliva (2014). In fMRI we computed *one RDM per region*; in MEG/EEG we compute *one RDM per timepoint*, and we close by fusing the two modalities.

We will use **Python** 🐍 with [MNE-Python](https://mne.tools/) for the EEG side and [`rsatoolbox`](https://rsatoolbox.readthedocs.io/) (already used in sess-2) for the RSA side.

> **MEG vs EEG, in one sentence.** MEG records magnetic fields with SQUID sensors above the scalp; EEG records voltage differences between scalp electrodes. Both are time-resolved, sensor-level measurements, and the analysis pipelines (filter, bad-channel detection, epoching, ERP/ERF averaging, time-resolved RSA) are essentially identical. We use **EEG** in §1 to §4 (the system most labs actually have) and pivot to a **published MEG dataset** in §5 to be able to fuse our results back into the fMRI work from sess-2.

### Practical advice

  - **Read the markdown cells.** The code cells are the recipe; the text says what is being cooked and why.
  - **Run the cells in order.** Each cell builds on the previous one. Skipping ahead will make later cells fail.
  - **Don't rush past the ❓ questions.** Try to answer each one in your head first; model answers are kept separately, not in this notebook.


## 0. Setup 🐍

The cell below installs the libraries we need (only on first run) and imports them.

  - **mne**, EEG/MEG IO, preprocessing, plotting.
  - **mne-bids**, read BIDS-formatted EEG with one call.
  - **pybids**, query a BIDS dataset like a database (already used in sess-1a/1b/2).
  - **rsatoolbox**, the RSA library you already met in sess-2.
  - **h5py**, reads MATLAB v7.3 `.mat` files (the Algonauts releases use this format).


In [ ]:
# `%pip` is the notebook magic for installing Python packages into the
# current kernel (same as in sess-1b and sess-2). We need:
#   - mne / mne-bids: M/EEG IO and preprocessing, BIDS reader
#   - pybids        : query a BIDS dataset like a database (already in sess-1a)
#   - rsatoolbox    : representational similarity analysis (already in sess-2)
#   - h5py          : reads MATLAB v7.3 .mat files (used in §5)
#   - pyprep        : PREP pipeline NoisyChannels detector (used in §3.3)
%pip install -q mne mne-bids pybids rsatoolbox h5py pyprep

import numpy as np
import pandas as pd                  # tabular data (events.tsv, channel tables)
import matplotlib.pyplot as plt
import mne                           # MNE-Python, the M/EEG library
import rsatoolbox                    # RSA library used in §5
from pathlib import Path             # object-oriented file paths

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)
mne.set_log_level("WARNING")          # silence the chatter; "INFO" if curious

print("MNE        version :", mne.__version__)
from importlib.metadata import version
print("rsatoolbox version :", version("rsatoolbox"))


### Where the data lives

Same convention as sess-1b and sess-2: the shared course folder lives at `/data/teaching/costantinoai/sess-3/` and is **read-only**. The cell below copies it into `results/sess-3/` under your clone (~75 MB, a few seconds), and the rest of the notebook reads from there.

  - `sub-01/eeg/` is the **EEG** dataset students preprocess in §2 to §4: one run from one subject of [OpenNeuro **ds003061**](https://openneuro.org/datasets/ds003061) (Delorme, "EEG data from an auditory oddball task", CC0). It is **truly raw**, no upstream filters applied, so the PSD in §2 will show all the textbook features (1/f, alpha, mains line) and §3.2 has real cleaning to do.
  - `derivatives/cichy_2014/` holds the Cichy 2014 stimuli + a small info-only fif carrying the **Neuromag 306-channel MEG layout** that the §5 theoretical section uses for its topomap demo.
  - `derivatives/algonauts_2019/` holds the **group-level fMRI and MEG RDMs** released as part of the Algonauts 2019 challenge. They are what we close §5 with.


In [ ]:
%%bash
# Copy the sess-3 data from the shared course folder into your personal
# results/. The share is read-only; results/sess-3/ is yours to read and write.
set -e
mkdir -p results/sess-3
rm -rf results/sess-3/*
cp -r /data/teaching/costantinoai/sess-3/* results/sess-3/
chmod -R u+w results/sess-3
echo "==== top-level layout of results/sess-3/ ===="
find results/sess-3 -maxdepth 2 -not -path "*/derivatives/*" -not -path "*/eeg" -not -path "*/cichy_2014" -not -path "*/algonauts_2019" -not -path "*/kriegeskorte_2008" | sort
echo
echo "==== EEG run files ===="
ls results/sess-3/sub-01/eeg/ | sed 's/^/  /'

In [ ]:
RESULTS_DIR = Path("results/sess-3")
BIDS_ROOT   = RESULTS_DIR
DERIV_DIR   = RESULTS_DIR / "derivatives"
CICHY_DIR   = DERIV_DIR / "cichy_2014"
ALGO_DIR    = DERIV_DIR / "algonauts_2019"
KRIEG_DIR   = DERIV_DIR / "kriegeskorte_2008"

# Sanity-check the copy. If any of these asserts fires, re-run the bash cell.
assert (BIDS_ROOT / "sub-01" / "eeg").exists(), "EEG missing; re-run the copy cell at the top"
assert (CICHY_DIR / "cichy_meg_info.fif").exists(), CICHY_DIR
assert (ALGO_DIR / "target_fmri.mat").exists(), ALGO_DIR
print("Reading data from :", BIDS_ROOT.resolve())


## 1. The dataset and the experiment

The EEG side of today's session uses **one subject from [OpenNeuro ds003061](https://openneuro.org/datasets/ds003061)** (Delorme, 2020, *"EEG data from an auditory oddball task"*, CC0), recorded at the Meditation Research Institute in Rishikesh, India. The paradigm is the **classical auditory oddball**:

  - The participant sat with **eyes closed**, a keypad on their lap.
  - Through headphones they heard a stream of three sound types, ~1 per second:
    - **Standard** tone (500 Hz pure tone, 60 ms): **70%** of stimuli.
    - **Oddball** tone (1000 Hz pure tone, 60 ms): **15%** of stimuli, *targets*.
    - **Distractor** white noise (60 ms): **15%** of stimuli, *non-targets*.
  - Task: press the keypad on every **oddball**, ignore the rest.

The classical EEG signature of this task is the **P300 / P3b**: a positive bump on **central-parietal** electrodes (`Pz`, `CPz`), peaking around **300 to 500 ms** after a target/oddball, much larger than the response to standards. It indexes the brain's "I detected the rare, task-relevant event" response and is one of the best-studied ERP components.

> 💡 **Why eyes closed?** The protocol asked participants to keep their eyes closed throughout. That has two side-effects you will see in the PSD in §2: (1) a strong **alpha rhythm** (8 to 12 Hz) over posterior scalp, the classical "Berger rhythm" that appears within seconds of closing the eyes; (2) very few blink artefacts (no saccades).

### The recording

| Property | Value |
|---|---|
| EEG system | BioSemi Active 2 |
| Scalp electrodes | **64** (10-10 layout, FP1, AF7, AF3, F1, ..., O2) |
| Other channels | 11 misc + 2 GSR + 1 respiration + 1 temperature (we drop these in §3.1) |
| Sampling rate | **256 Hz** |
| Mains frequency | **50 Hz** (recording in India) |
| Pre-applied filters | **None** (data is genuinely raw, no software filters applied upstream) |
| Run duration | ~12.6 minutes (1 of 3 runs staged here; runs 2 and 3 are on OpenNeuro) |
| Trials per run | ~750 stimuli (525 standards, ~113 oddballs, ~112 distractors) |
| File format | BIDS-EEG with **EEGLAB** container (`.set`) |


> 💡 **Visual summary, what gets presented**
>
> ![Auditory oddball task overview](assets/sess-3/figures/task_overview.png)
>
> Top: a 10-second slice of the actual stimulus stream. Each vertical bar is one stimulus, colour-coded by type; with 70 % standards in the mix, you should see ~7 blue bars per 10 s and ~1 to 2 of each minority type.
> Bottom: the three sound types at audio level. The two tones differ by an octave (500 Hz vs 1000 Hz, easy to discriminate), and the distractor is a 60 ms burst of white noise.
>
> **Procedure.** Each stimulus lasts 60 ms with 5 ms cosine ramps; the three types are interleaved randomly with about 1 s between them. The participant sat with their **eyes closed** for the whole run, a keypad on their lap, and was told to press the keypad on every **oddball** and ignore the rest.


### 1.1 A note on BIDS

The dataset on disk follows the same BIDS structure you met in sess-1a, sess-1b, and sess-2, here under `sub-01/eeg/` with the usual sidecars (`.json`, `channels.tsv`, `events.tsv`, `electrodes.tsv`). In a typical analysis you would query that layout with [`pybids`](https://bids-standard.github.io/pybids/) (`BIDSLayout(...).get(...)`) so paths are not hard-coded, exactly as in sess-1a §3 and sess-1b §3.

Here we have one subject and one run, so we skip the query step and point [`mne_bids.read_raw_bids`](https://mne.tools/mne-bids/stable/generated/mne_bids.read_raw_bids.html) directly at the file in §2. The reader still picks up the sidecar JSON (sampling rate, channel types, line frequency) and the events from `events.tsv` automatically.


## 2. A first look at the raw EEG 🐍

We use [`mne_bids.read_raw_bids`](https://mne.tools/mne-bids/stable/generated/mne_bids.read_raw_bids.html) to load the recording with a single call: it picks up the sidecar JSON (sampling rate, channel types, electrode positions, line frequency) and the events from the `events.tsv` automatically.


In [ ]:
# Use mne_bids to load the BIDS-EEG recording with one call: it picks up the
# sidecar JSON (sampling rate, channel types, line freq) and the events from
# events.tsv automatically.
# docs:      https://mne.tools/mne-bids/stable/generated/mne_bids.read_raw_bids.html
# signature: read_raw_bids(bids_path, extra_params=None, verbose=None)
from mne_bids import BIDSPath, read_raw_bids

# A `BIDSPath` is the canonical way to point at a file in a BIDS dataset; it
# auto-builds the filename from the entities (subject, task, run, ...).
# docs: https://mne.tools/mne-bids/stable/generated/mne_bids.BIDSPath.html
bids_path = BIDSPath(
    subject="01",
    task="P300",
    run="1",
    datatype="eeg",
    suffix="eeg",
    extension=".set",          # EEGLAB container (single-file)
    root=BIDS_ROOT,
)
raw = read_raw_bids(bids_path, verbose="error")
raw.load_data()                # pull the binary signal into memory

# The electrode positions in the BIDS sidecar use a non-MNE coordinate
# convention (Lz/Fpz are flipped, T7/T8 land in the wrong front/back), which
# would render scalp topographies upside-down or mirrored. Override with the
# canonical BioSemi 64-channel 10-10 montage; this dataset was recorded on a
# standard BioSemi Active 2, so the layout matches exactly.
# docs: https://mne.tools/stable/generated/mne.channels.make_standard_montage.html
montage = mne.channels.make_standard_montage("biosemi64")
raw.set_montage(montage, on_missing="ignore", match_case=False)
print(raw)


In [ ]:
# Quick descriptive stats.
sfreq    = raw.info["sfreq"]
n_chan   = raw.info["nchan"]
duration = raw.times[-1]
ch_types = pd.Series(raw.get_channel_types()).value_counts()

print(f"Sampling frequency : {sfreq:.0f} Hz")
print(f"Number of channels : {n_chan}")
print(f"Recording duration : {duration:.1f} s ({duration/60:.1f} min)")
print()
print("Channel types:")
print(ch_types.to_string())


> ❓ **Question.** 
> 
> In the output of the cell above, you can see something like `<RawEEGLAB | sub-01_task-P300_run-1_eeg.set, 79 x 194048 (758.0 s), ...>`. What do these numbers represent exactly? And what would they look like if we recorded data with a **128-channel** EEG system at **512 Hz** for **100 seconds**?


### Sensor layout

The BIDS sidecar `sub-01_task-P300_run-1_electrodes.tsv` carries the 3D electrode positions; `read_raw_bids` builds a montage from it automatically. Plot the sensors to confirm.


In [ ]:
# Plot the electrode topomap to confirm the BIDS sidecar electrodes.tsv was
# read correctly. Each dot is one electrode at its scalp position; with
# show_names=True you see the standard 10-10 labels.
# docs:      https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.plot_sensors
# signature: raw.plot_sensors(kind='topomap', ch_type='eeg', show_names=False, ...)
fig = raw.plot_sensors(kind="topomap", show_names=True, ch_type="eeg", show=False)
fig.suptitle("ds003061 BioSemi 64-channel layout", y=1.02)
plt.tight_layout()
plt.show()


### Power spectral density

The **PSD** is the cheapest first sanity check: it tells us at a glance which frequencies carry energy and whether anything looks dramatically wrong. In a typical raw EEG recording you should see:

  - A **1/f-like background**, energy falling off with frequency on a log-log plot.
  - A **prominent alpha peak** around **8 to 12 Hz**, especially over posterior electrodes (the back of the head). This dataset was recorded with the participant's **eyes closed**, so the alpha rhythm should be unmistakable.
  - A **sharp narrow spike at the mains frequency**. This recording was made in **India**, where mains is **50 Hz**.

All three features should be visible in the figure below.

> 💡 **Why eyes closed makes alpha pop.** The "Berger rhythm" in alpha (8-12 Hz, posterior) is the brain's idle-mode rhythm of visual cortex; with eyes closed, visual cortex is not engaged in pattern processing and oscillates in synchrony at alpha frequency. With eyes open, alpha is suppressed (event-related desynchronisation). This is the very first phenomenon ever documented in human EEG, by Hans Berger in 1929.


In [ ]:
# Power Spectral Density: amplitude per frequency, averaged across the whole
# recording. The cheapest first sanity check, confirms brain rhythms sit on
# a 1/f-like background and shows mains hum (50/60 Hz) as a sharp spike.
# docs:      https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.compute_psd
# signature: raw.compute_psd(fmin=0, fmax=inf, picks=None, n_jobs=1, ...)
#
# Suppress MNE's default vertical gray dashed marker at the line frequency
# (50 Hz here): we already know where mains is, the marker just clutters.
raw.info["line_freq"] = None

spectrum = raw.compute_psd(fmax=120, n_jobs=-1)
fig = spectrum.plot(picks="eeg", show=False)
fig.suptitle("Raw PSD across all EEG sensors", y=1.02)
plt.tight_layout()
plt.show()

# Spot the odd-one-out trace. Notice the light-green line sitting well below
# the bundle (~ -30 to -45 dB while the rest are between -20 and +20 dB),
# with a regular comb of peaks across the whole spectrum. That is the
# fingerprint of a disconnected / floating electrode: total broadband power
# is anomalously low, and what little signal there is comes from
# environmental EM pickup at fixed frequencies. Rank channels by mean log
# power and the outlier should jump out at the bottom of the list.
psd_data, freqs = spectrum.get_data(return_freqs=True)
mean_log_power  = np.log10(psd_data + 1e-30).mean(axis=1)
order           = np.argsort(mean_log_power)   # low power first
print("Five lowest-power channels (likely disconnected / floating):")
for r in order[:5]:
    print(f"  {spectrum.ch_names[r]:6s}  mean log10(power) = {mean_log_power[r]:6.2f}")


> ❓ **Question.** Look at the PSD above. (1) Where is the alpha peak, and how strong is it relative to neighbouring frequencies? (2) Where is the mains spike? (3) What does the slope of the curve look like at very low frequencies (< 5 Hz), and what would you do about it before computing ERPs?


## 3. Preprocessing

Before we can analyse the EEG response to the auditory stimuli we need to clean and reshape the raw signal. Preprocessing aims to **reduce noise sources while keeping the signal that is relevant for the experiment**.

| # | What we do | MNE call | Why |
|---|---|---|---|
| 3.1 | Pick EEG channels (drop GSR, respiration, temperature, misc) | `raw.pick("eeg")` | We only want to analyse scalp EEG for the ERP; no auxiliary sensors here |
| 3.2 | High-pass filter at 0.1 Hz | `raw.filter(0.1, None)` | Removes very slow drifts (sweat, breathing, electrode settling) before bad-channel detection |
| 3.3 | Detect and interpolate bad channels | `pyprep.NoisyChannels`, `interpolate_bads()` | A noisy channel triggers our peak-to-peak rejection on most trials and corrupts the average reference. Run on highpassed-only data (PREP convention). |
| 3.4 | Low-pass filter at 40 Hz | `raw.filter(None, 40)` | Removes high-frequency noise (muscle, mains) once the bad channels are out of the way |
| 3.5 | Re-reference to the average | `raw.set_eeg_reference("average")` | EEG voltages are relative; the average reference is the standard ERP convention |
| 3.6 | Find stimulus onsets, epoch, reject blink-y trials | `events_from_annotations`, `mne.Epochs` | Aligns trials to stimulus onset and removes the worst artefacts before averaging |



Normally the data would also be downsampled from e.g., 1024 Hz to 256 Hz. Here, the recording is already at 256 Hz, so we skip the downsampling.


### 3.1 Pick EEG channels 🐍

The dataset has **79 channels**: 64 EEG, plus auxiliary channels (GSR, respiration, temperature, misc) recorded by the BioSemi system. We only want the 64 EEG channels for the ERP and RSA work below; the auxiliaries are useful for other questions but distract here.

Note that this particular dataset does **not** include EOG (electro-oculography) channels. EOG is useful to remove eye movements artifacts: when eyes move during a stimulus presentation, they create a big artifact in th signal that can affect the qualirty of your data. By using the EOG signal, we know when the eyes moved, and this can be useful to dismiss some trials or regress out (like we did in the fMRI session) eye-movements from the data. 

With the participant's eyes closed throughout the recording, blinks are nearly absent here, so the protocol simply did not deploy EOG sensors. In datasets where EOG is recorded (e.g. ERP CORE), the convention is to mark them as `eog` channel type with `raw.set_channel_types(...)` so MNE knows to use them for blink-aware artefact handling.


In [ ]:
# Confirm what we have and pick only the 64 EEG channels (drop GSR,
# respiration, temperature, and misc auxiliary channels). `pick` modifies
# `raw` in place and returns it for chaining.
# docs:      https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.pick
# signature: raw.pick(picks)        — picks can be a list of names, types, or both
print("Channel types BEFORE picking:")
print(pd.Series(raw.get_channel_types()).value_counts().to_string())

raw.pick("eeg")                # keep only EEG channels (64 of them)
print()
print("Channel types AFTER picking:")
print(pd.Series(raw.get_channel_types()).value_counts().to_string())


### 3.2 High-pass filter at 0.1 Hz 🐍

We apply only the **high-pass** here, leaving the low-pass for §3.4. The high-pass at 0.1 Hz removes very slow drifts (sweat under the electrodes, breathing, the cap settling) without touching the rest of the band.

> 💡 **Why split the band-pass?** PREP's bad-channel detectors (§3.3) need the full band above the highpass to do their job: `bad_by_hf_noise` works by computing per-channel HF/LF power ratios, and if we lowpass at 40 Hz first, we drive every channel into the filter's stopband and the test starts measuring numerical noise instead of real high-frequency contamination. We highpass now, run the bad-channel detection on the result, and apply the 40 Hz lowpass afterwards.


In [ ]:
# High-pass the continuous EEG. MNE's default is a zero-phase FIR with
# `firwin` design and Hamming window; n_jobs parallelises across channels.
# Pass h_freq=None to skip the low-pass (we apply that in §3.4).
# docs:      https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.filter
# signature: raw.filter(l_freq, h_freq, picks=None, n_jobs=1, ...)
raw.filter(
    l_freq=0.1,         # high-pass: remove drifts < 0.1 Hz
    h_freq=40,        # no low-pass yet, see §3.4
    n_jobs=-1,
    verbose="error",
)

### 3.3 Spotting and interpolating bad channels 🐍

Real EEG recordings almost always have one or two electrodes that misbehave the entire session, perhaps a wire was loose, the gel dried out, or the electrode never made good contact with the scalp. If we leave them in:

  1. they trigger our artefact rejection in on most trials, throwing away huge amounts of data;
  2. they corrupt the **average reference**, smearing one channel's problem across all the others.

We mark them as `bads`, then interpolate them from neighbours: pretend the bad electrode wasn't there and use the weighted average of nearby good electrodes to estimate what it would have measured. This works because EEG signal of nearby electrodes tends to be correlated, due to the spatial distribution of electrical current on the scalp.

**Detection.** A bad channel can fail in several different ways: dead and silent, drifting, full of high-frequency hash, decorrelated from its neighbours, etc. No single rule catches all of these, so the [PREP pipeline](https://doi.org/10.3389/fninf.2015.00016) runs **several independent detectors** and flags the union. We use [`pyprep.NoisyChannels`](https://pyprep.readthedocs.io/en/latest/generated/pyprep.NoisyChannels.html), which exposes seven of them: flat, deviation, high-frequency noise, correlation with neighbours, SNR, dropout, and RANSAC. 


In [ ]:
# Run the PREP pipeline's NoisyChannels detector. It applies seven
# independent tests and exposes which channels each one flagged; we take
# the union and pass it to interpolate_bads.
# docs: https://pyprep.readthedocs.io/en/latest/generated/pyprep.NoisyChannels.html
from pyprep.find_noisy_channels import NoisyChannels

nd = NoisyChannels(raw, random_state=0)
nd.find_all_bads()

bad_chans = nd.get_bads()
print(f"\nBad channels40: {bad_chans or '(none)'}")

raw.info["bads"] = bad_chans
if bad_chans:
    # docs: https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.interpolate_bads
    raw.interpolate_bads(reset_bads=True, verbose="error")
    print("Bad channels interpolated.")


> ❓ **Question.** Why does a genuine brain signal usually appear on **several neighbouring electrodes** rather than only one?


### 3.4 Low-pass filter at 40 Hz 🐍

Now that the bad channels are out, we apply the low-pass at 40 Hz: removes high-frequency noise (muscle, mains hum at 50 Hz). ERPs live well below 40 Hz, so everything above that is not signal of interest.

> 💡 **Why now?** Doing it after §3.3 means the bad-channel detectors saw the full spectrum and could compute meaningful HF/LF ratios. Doing it before re-referencing in §3.5 means the average reference is computed on the same band the ERP analysis will use.


In [ ]:
# Low-pass at 40 Hz. Removes muscle hash and the 50 Hz mains spike.
# docs: https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.filter
raw.filter(
    l_freq=None,        # high-pass already done in §3.2
    h_freq=40.0,        # low-pass: remove >40 Hz (incl. 50 Hz mains)
    n_jobs=-1,
    verbose="error",
)

### 3.5 Re-referencing to the average 🐍

EEG measures **voltage differences** between electrodes. There is no absolute zero on the head; every recording is made relative to some chosen reference (often a mastoid or the system reference). For ERP analysis the standard choice is the **average reference**: at every time point, subtract from every electrode the average across all electrodes at that time. So now our zero is the average activity across all electrodes, and positive/negative peaks reflect the variation from that average.

> 💡 **Why now?** We re-reference *after* interpolating bad channels so the bad ones do not poison the average. If we did it before, the (now-noisy) average would smear the bad channel's problems across all the good channels.


In [ ]:
# Subtract from every electrode the average across all electrodes at every
# time point. The "common" choice for ERP analysis. `projection=False` makes
# the change immediate and irreversible (vs. stored as a projector to apply
# later).
# docs:      https://mne.tools/stable/generated/mne.io.Raw.html#mne.io.Raw.set_eeg_reference
# signature: raw.set_eeg_reference(ref_channels='average', projection=False,
#                                   ch_type='auto', forward=None, ...)
raw.set_eeg_reference(ref_channels="average", projection=False, verbose="error")
n_eeg = sum(t == "eeg" for t in raw.get_channel_types())
print(f"Re-referenced to the average of all {n_eeg} EEG electrodes.")


> ❓ **Question.** The P300 is a *positive* bump on **central-parietal** electrodes (around `Pz` / `CPz`). After average-referencing, what should you expect to see at distant electrodes (e.g. very lateral or frontal) at the same time?


### 3.6 Find events, epoch, reject 🐍

Note that we **do not need a separate downsample step** here: the recording is already at 256 Hz, well above what ERPs need (the 40 Hz low-pass leaves the highest meaningful frequency at 40 Hz, so any sampling rate above 80 Hz would be enough by Nyquist). If your own data ever lands at a higher rate, the call would be `raw.resample(256, n_jobs=-1)`.

`read_raw_bids` already converted the BIDS events.tsv into MNE annotations on `raw`. We extract them as an `events` array and re-build the `event_id` dictionary, then **slice** the recording into trials, one slice per stimulus (an **epoch**).

We cut from **−200 ms** to **+800 ms** around each tone:

  - The 200 ms *before* the tone is the **baseline**, brain idle activity. We subtract its mean so every epoch starts at 0 µV.
  - The 800 ms *after* covers the response window where the P300 lives.

We also **drop any trial where any EEG channel exceeds 100 µV peak-to-peak**. Eyes-closed data has fewer blinks than eyes-open, so this threshold catches mostly muscle and movement artefacts.


In [ ]:
# ds003061 has two quirks we normalise here:
#   (1) "oddball_with_reponse" / "noise_with_reponse" should sit *inside* the
#       oddball/noise hierarchy so MNE's slash selector picks them up via
#       epochs["oddball"]. We split the underscore into a slash.
#   (2) "reponse" in the source data is a typo; we keep it for now but the
#       split places it after the slash where it does not affect selection.
new_desc = []
for d in raw.annotations.description:
    d = d.replace("oddball_with_reponse", "oddball/with_response")
    d = d.replace("noise_with_reponse",   "noise/with_response")
    new_desc.append(d)
raw.set_annotations(mne.Annotations(
    onset       = raw.annotations.onset,
    duration    = raw.annotations.duration,
    description = new_desc,
    orig_time   = raw.annotations.orig_time,
))

# Filter to stimulus annotations only with the `regexp` arg; the `response`
# annotation is not stimulus-locked and would land at a different sample.
# After the rename above, the event_id keys are:
#   stimulus/standard                 (frequent tone)
#   stimulus/oddball                  (rare target, no button press, very few)
#   stimulus/oddball/with_response    (rare target, button pressed)
#   stimulus/noise                    (distractor, no press, very few)
#   stimulus/noise/with_response      (distractor, false alarm)
events, event_id = mne.events_from_annotations(raw, regexp="^stimulus/",
                                                verbose="error")
event_id = {str(k): int(v) for k, v in event_id.items()}
print(f"Number of stimulus events: {len(events)}")
print("event_id (stimulus only):")
for k, v in event_id.items():
    print(f"  {v:3d}  {k}")


In [ ]:
# Slice the continuous recording into trials, baseline-correct, drop blink-y
# trials. The slash-separated event ids (`stimulus/standard`,
# `stimulus/oddball`, `stimulus/noise`, ...) let us later select with
# epochs["standard"], epochs["oddball"], epochs["noise"].
# docs:      https://mne.tools/stable/generated/mne.Epochs.html
# signature: mne.Epochs(raw, events, event_id=None, tmin=-0.2, tmax=0.5,
#                       baseline=(None, 0), picks=None, preload=False,
#                       reject=None, flat=None, proj=True, decim=1, ...)
epochs = mne.Epochs(
    raw,
    events=events,
    event_id=event_id,
    tmin=-0.2,                              # 200 ms pre-stimulus baseline
    tmax=0.8,                               # 800 ms post-stimulus
    baseline=(None, 0),                     # subtract mean of [start..0] from each trial
    reject=dict(eeg=100e-6),                # peak-to-peak rejection: 100 µV per EEG ch
    preload=True,                           # load all epochs into RAM (small dataset)
    on_missing="ignore",                    # event_ids never observed: skip silently
    verbose="error",
)
print(epochs)
print(f"\nKept {len(epochs)} stimulus epochs after peak-to-peak rejection.")

# Counts per condition. Note that "oddball" matches BOTH `stimulus/oddball`
# and `stimulus/oddball_with_reponse` because of MNE's slash notation.
print()
print("Trials per condition:")
print(f"  standard  : {len(epochs['standard'])}")
print(f"  oddball   : {len(epochs['oddball'])}")
print(f"  noise     : {len(epochs['noise'])}")

del raw


> ❓ **Question.** Why do we keep a 200 ms baseline **before** the stimulus? Why not just start at t = 0?


## 4. The event-related potential (ERP) 🐍

An **event-related potential** is what you get when you take many trials of the same condition and **average them, time point by time point**. The average is the cleaned-up, repeatable response of the brain to that type of stimulus. Anything not time-locked to the stimulus (random alpha oscillations, ongoing background activity) cancels out across trials. What survives is the **stimulus-locked signal**.

We compute three averages:

  - **all stimuli pooled**, the textbook auditory evoked response (N1/P2 around 100/200 ms).
  - **standard tones only**, the response to expected, frequent stimuli.
  - **oddball tones only**, the response to rare, task-relevant stimuli.

The contrast between **oddball and standard** isolates the **P300 / P3b**: a positive bump on **central-parietal** electrodes, peaking ~300 to 500 ms after a target, much larger for oddballs than for standards. It is one of the best-studied ERP components and indexes the brain's "I detected the rare, task-relevant event" response.


### 4.3 Standard vs oddball: the P300

Average each condition separately and contrast them. The **P300** should be visible as a **larger positivity for oddball trials** at central-parietal electrodes (around `Pz` / `CPz`) ~300 to 500 ms post-stimulus.

> 💡 **Why does this contrast work?** The oddball is rare *and* task-relevant (the participant must press a button). Both rarity and task-relevance contribute to the P300; the textbook decomposition is **P3a** (frontal-central, ~250 to 300 ms, novelty / attention) and **P3b** (parietal, ~300 to 500 ms, target detection / context updating). With eyes-closed BioSemi 64-ch, the P3b dominates and is what you will see most clearly.

> 💡 **Why fewer oddball trials makes the oddball average noisier.** ~520 standards vs ~113 oddballs is roughly a 5× difference in trial count; ERP noise scales as 1/√N, so the oddball average has ~√5 ≈ 2.2× more noise than the standard average. The P300 is large enough to survive that anyway, but it is one reason oddball studies often record many runs.


In [ ]:
# Standard vs oddball evokeds.
# combine_evoked with weights=[1,-1] returns the difference wave.
# docs: https://mne.tools/stable/generated/mne.combine_evoked.html
evk_standard = epochs["standard"].average()
evk_oddball  = epochs["oddball"].average()
diff         = mne.combine_evoked([evk_oddball, evk_standard], weights=[1, -1])
diff.comment = "oddball - standard"

# Pick a central-parietal electrode for the comparison (canonical P300 site).
# Falls back to Cz if Pz/CPz are not in the montage.
parietal_chan = next((ch for ch in ("Pz", "CPz", "Cz") if ch in epochs.ch_names),
                      epochs.ch_names[0])

# Overlay the two evokeds at one channel; very useful for showing condition
# differences at a single sensor of interest.
# docs:      https://mne.tools/stable/generated/mne.viz.plot_compare_evokeds.html
# signature: plot_compare_evokeds(evokeds, picks=None, colors=None, linestyles=None,
#                                  styles=None, cmap=None, vlim=(None, None), ci=None, ...)
fig = mne.viz.plot_compare_evokeds(
    {"standard": evk_standard, "oddball": evk_oddball},
    picks=parietal_chan, show=False,
    title=f"Standard vs oddball at {parietal_chan}",
)
plt.show()


### 4.4 Joint plot: time course + topographies

`plot_joint` is the most information-dense visualisation in MNE: a butterfly at the bottom, topographies of the most interesting time points on top. We plot the **difference** wave (oddball − standard) so the P300 jumps out.


In [ ]:
# Most information-dense MNE figure: a butterfly plot at the bottom plus
# topographies at chosen latencies on top. We pass three latencies of
# pedagogical interest (N1, P3a, P3b).
# docs:      https://mne.tools/stable/generated/mne.Evoked.html#mne.Evoked.plot_joint
# signature: evoked.plot_joint(times='peaks', title=None, picks=None,
#                              exclude='bads', show=True, ts_args=None,
#                              topomap_args=None)
fig = diff.plot_joint(picks="eeg",
                      times=[0.150, 0.300, 0.450],
                      title="Oddball - Standard",
                      show=False)
plt.show()


> ❓ **Question.** Where on the scalp is the difference largest in the topomaps above? Does that match the textbook description of the P300 generator?


## 5. Representational similarity analysis (RSA)

RSA was the closing topic of [sess-2 §3](sess-2.ipynb): we built one **representational dissimilarity matrix (RDM)** per visual ROI (EVC, IT) on the Kriegeskorte 92-image fMRI data and showed that EVC and IT carry distinct stimulus geometries.

Today we pick up the **MEG counterpart** of that same experiment (Cichy, Pantazis & Oliva, 2014: same 92 stimuli, same 16 subjects' worth of brain data) and use it to do something fMRI alone cannot: **track the geometry over time and fuse it back into the fMRI ROI RDMs**.

### 5.1 The shape of the question changes when the modality changes 

EEG and MEG do not measure the brain region by region; they measure it **millisecond by millisecond**. The natural unit of analysis is therefore not "the pattern across voxels in this ROI" but "the pattern across all sensors at this single time point". So the central object of the analysis flips: **one RDM per region** in fMRI becomes **one RDM per millisecond** in MEG/EEG. The fMRI bar chart from sess-2 §3.6 becomes a **time course**: at every millisecond, how well does the MEG geometry match the fMRI geometry of EVC and of IT?

> 💡 **Why this connects fMRI and MEG.** fMRI tells us *where* a representation lives but not *when* (BOLD is too slow); MEG/EEG tells us *when* but not *where* (sensors are far from the sources, the inverse problem is hard). RSA is the trick that puts them on a common axis: an RDM is the same shape whether it comes from voxels in IT or sensors at 150 ms. Once both are 92 × 92 matrices, you can ask *at what time does the MEG geometry look most like the fMRI IT geometry?*. That is the basis of **MEG-fMRI fusion** (Cichy, Pantazis & Oliva, 2014, 2016).


In [ ]:
# Load Kriegeskorte category vectors and stimulus thumbnails (same as sess-2).
# These give us:
#   animate    : (92,) binary array, 1 if the stimulus depicts an animate object
#   group_lab  : (92,) array of 'animate' / 'inanimate' strings, used to reorder
#                rows in show_rdm so the categorical block jumps out
#   conds      : (92,) array of unique stimulus keys, required by calc_rdm
#   icons      : (92,) Icon objects used as axis tick labels on the RDMs
from scipy.io import loadmat
from rsatoolbox.vis.icon import Icon

supp = loadmat(KRIEG_DIR / "Kriegeskorte_Neuron2008_supplementalData.mat",
               simplify_cells=True)
stimuli         = supp["stimuli_92objs"]
category_labels = list(supp["categoryLabels"])
category_mat    = supp["categoryVectors"]

animate   = category_mat[:, category_labels.index("animate")].astype(int)
group_lab = np.where(animate == 1, "animate", "inanimate")
conds     = np.array([f"img_{i:02d}" for i in range(len(stimuli))])
icons     = np.array([Icon(image=s["image"], make_square=True) for s in stimuli])
print(f"92 stimuli: {animate.sum()} animate, {(1-animate).sum()} inanimate")


### 5.2 Load the Cichy 2014 group MEG RDMs 🐍

We mirror the per-subject MEG RDMs from Cichy 2014 locally at `derivatives/cichy_2014/MEG_decoding_RDMs.mat`. The shape is

  - **16 subjects × 2 sessions × 1301 timepoints × 92 × 92**, with timepoints sampled every 1 ms from −100 ms to +1200 ms post-stimulus.

The dissimilarity measure here is **pairwise decoding accuracy**: a cross-validated linear classifier (SVM) trained to distinguish stimulus *i* from stimulus *j* on the per-trial sensor patterns at that timepoint. High accuracy (~100%) means the two stimuli are easy to tell apart, low accuracy (~chance = 50%) means their patterns are similar. *Same logic, different distance metric*: in sess-2 the fMRI dissimilarity was 1 − r; here it is decoding accuracy. RSA is metric-agnostic, the per-timepoint matrix is still 92 × 92 and still comparable across modalities.

> 💡 **Why decoding accuracy?** It is more robust than 1 − r when sensor patterns are noisy and trial counts are modest, because the classifier can learn a discriminative projection rather than relying on raw correlation. Most modern MEG/EEG RSA pipelines use it.


In [ ]:
# Load the Cichy 2014 group MEG RDMs.  The .mat file is HDF5 v7.3 so we use
# h5py.  h5py reads MATLAB axes reversed, so the published shape
# (16 subj, 2 sess, 1301 t, 92, 92) appears as (92, 92, 1301, 2, 16).
# The full array is ~2.8 GB; loading takes ~10 s on Neurodesk.
import h5py

with h5py.File(CICHY_DIR / "MEG_decoding_RDMs.mat", "r") as f:
    meg_full = np.array(f["MEG_decoding_RDMs"]).transpose((4, 3, 2, 1, 0))
print("MEG RDMs full :", meg_full.shape, "(subj, sess, time, 92, 92)")

# Patch the NaN diagonal up front so np.nanmean does not warn about empty
# slices, and so rsatoolbox can wrap the arrays cleanly.
np.einsum("...ii->...i", meg_full)[...] = 0.0

# Average across the two sessions, keep the per-subject time course.
# Shape after: (16 subj, 1301 t, 92, 92).
meg_subj_time = meg_full.mean(axis=1)
del meg_full

# Group-mean RDM per timepoint, shape (1301, 92, 92).
meg_group = meg_subj_time.mean(axis=0)
print("Group MEG :", meg_group.shape, "(time, 92, 92)")
print(f"Decoding accuracy range: {meg_group.min():.1f}% to {meg_group.max():.1f}%")

# Time axis: 1301 samples from -100 to +1200 ms in 1 ms steps.
meg_times_ms = np.linspace(-100, 1200, 1301)

# Wrap the group MEG RDMs once into a rsatoolbox.RDMs object with a `time`
# descriptor; everything below uses this single object instead of re-wrapping.
# `time` is in seconds because rsatoolbox.vis.timecourse.plot_timecourse
# expects seconds by default.
rdms_meg_group = rsatoolbox.rdm.RDMs(
    dissimilarities       = meg_group,
    dissimilarity_measure = "decoding accuracy",
    rdm_descriptors       = {"time": meg_times_ms / 1000.0},
    pattern_descriptors   = {"conds": conds, "icons": icons,
                             "group": group_lab},
)


In [ ]:

# Also load the matching fMRI group RDMs for §5.5.
with h5py.File(ALGO_DIR / "target_fmri.mat", "r") as f:
    fmri_it_mean  = np.transpose(np.array(f["IT_RDMs"]),  (2, 0, 1)).mean(axis=0)
    fmri_evc_mean = np.transpose(np.array(f["EVC_RDMs"]), (2, 0, 1)).mean(axis=0)

fmri_rdms = rsatoolbox.rdm.RDMs(
    dissimilarities       = np.stack([fmri_evc_mean, fmri_it_mean]),
    dissimilarity_measure = "1 - r",
    rdm_descriptors       = {"region": np.array(["EVC", "IT"])},
    pattern_descriptors   = {"conds": conds},
)
print("fMRI group RDMs:", fmri_rdms.dissimilarities.shape, "regions:", list(fmri_rdms.rdm_descriptors["region"]))


In [ ]:
# Zoom in on the IT peak (~200 ms) and plot that single RDM big, with
# stimulus thumbnails on the axes and animate stimuli reordered to the top.
# docs: https://rsatoolbox.readthedocs.io/en/stable/rsatoolbox.rdm.html#rsatoolbox.rdm.RDMs.subset
t_peak_ms = 273
t_idx     = int(np.argmin(np.abs(meg_times_ms - t_peak_ms)))
rdm_peak  = rdms_meg_group.subset("time", meg_times_ms[t_idx] / 1000.0)
rdm_peak  = rdm_peak.subset_pattern("group", ["animate", "inanimate"])

fig, _, _ = rsatoolbox.vis.show_rdm(
    rdm_peak,
    show_colorbar="panel",
    pattern_descriptor="icons",
    num_pattern_groups=5, gridlines=[-1], figsize=(11, 11),
)
fig.suptitle(f"Group MEG RDM @ {t_peak_ms} ms (animate stimuli on top)", y=0.93)
plt.show()


**What you should see.** The RDM at 250 ms (around the late IT response peak) shows clear **two-block structure**: a lighter (low-dissimilarity) animate-vs-animate block in the top-left, a lighter inanimate-vs-inanimate block in the bottom-right, a cluster for human faces, and darker (high-dissimilarity) animate-vs-inanimate cells off the diagonal. That is the categorical signature of the ventral visual stream, recovered from a single sensor-level recording.

> ❓ **Question.** The values in this RDM are **decoding accuracy in percent** (50 = chance, 100 = perfect). Why is "high accuracy" the right *dissimilarity* measure (so that easy-to-distinguish pairs get a *high* RDM cell), instead of "low accuracy"?


### 5.3 MEG-fMRI fusion 🐍

Final step: compare the **group MEG RDM at each timepoint** to the two **fMRI region RDMs** (EVC, IT) from sess-2. The brain itself is the predictor:

  - **fMRI EVC** is the geometry that early visual cortex carries on these 92 stimuli.
  - **fMRI IT**  is the geometry that high-level visual cortex carries on the same stimuli.
  - **MEG @ time t** is the geometry that the *whole brain* carries at millisecond t.

If the MEG curve for **fMRI EVC** peaks early and the curve for **fMRI IT** peaks late, the ventral visual stream is unfolding in time exactly as Cichy, Pantazis & Oliva (2014, 2016) predicted. That spatial-where-meets-temporal-when readout is what *fusion* buys us.


In [ ]:
# fmri_rdms and rdms_meg_group are already RDMs objects from §5.2.
# One vectorised compare(): shape (2 fMRI regions, 1301 timepoints).
fusion = rsatoolbox.rdm.compare(fmri_rdms, rdms_meg_group, method="spearman")
rho_to_evc, rho_to_it = fusion[0], fusion[1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(meg_times_ms, rho_to_evc, color="C2", lw=2, label="MEG vs fMRI EVC")
ax.plot(meg_times_ms, rho_to_it,  color="C4", lw=2, label="MEG vs fMRI IT")
ax.axvline(0, color="k", lw=0.6, ls="--", alpha=0.6, label="stim onset")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("Time post-stimulus (ms)")
ax.set_ylabel("Spearman ρ  (group MEG vs group fMRI)")
ax.set_title("MEG-fMRI fusion: when does MEG look like each fMRI region?")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"Best match to fMRI EVC : {meg_times_ms[np.argmax(rho_to_evc)]:.0f} ms (ρ = {rho_to_evc.max():.3f})")
print(f"Best match to fMRI IT  : {meg_times_ms[np.argmax(rho_to_it)]:.0f} ms (ρ = {rho_to_it.max():.3f})")


**The two curves cross.** Early (around 80 to 100 ms) the group MEG RDM is closer to **fMRI EVC**; late (around 150 to 250 ms) it is closer to **fMRI IT**. The ventral visual stream is unfolding in time, exactly as Cichy, Pantazis & Oliva (2014, 2016) predicted.

> 💡 The two modalities individually answer half the question: fMRI says *where* a representation lives but is too slow to say *when*; MEG says *when* a representation arises but its sensors are too far from the sources to say *where*. Putting MEG-at-time-t and fMRI-in-region-R on a shared 92×92 axis recovers **both axes**: low-level image structure lives in EVC and arrives first; categorical structure lives in IT and arrives later. This is the same multimodal fusion idea that came up in the lectures, applied at the *analysis* level rather than at the acquisition level.

> ❓ **Question.** The fMRI RDM for a region is *one matrix* (no time). The MEG RDM is *one matrix per millisecond*. If you wanted to make an fMRI RDM that also changed over time, what would you have to do at the experimental design level, and why is it usually not worth it?


## ❓ 7. Insight questions

The questions below test understanding rather than memorisation. Try to write a 2 to 4 sentence answer to each one before checking the model answers (kept separately, not in this notebook).

---

**Q1.** In sess-2 we computed **one fMRI RDM per ROI** (IT, EVC) and asked which model fitted each ROI. In sess-3 we computed **one MEG RDM per timepoint** and asked which model fitted each timepoint. *In your own words*, why does the natural unit of analysis flip from "region" to "timepoint" when we move from fMRI to MEG/EEG?

---

**Q2.** You record 30-channel EEG and notice one electrode is 30× noisier than its neighbours throughout the session. List **two reasons** why you must not just ignore it and proceed with analysis.

---

**Q3.** Sketch (in words) what the §5.5 MEG-fMRI fusion plot would look like for the **monkey IT RDM** from sess-2 §3.7 (which was based on 674 single-unit recordings, not BOLD), instead of the human fMRI IT RDM. Where on the time axis would you expect the peak to land, and what would it tell you?
